# Convert raw data to 'strict' json

In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json
import gzip
dataset_name = "Beauty"
os.makedirs(dataset_name, exist_ok=True)

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.dumps(eval(l))

# Beauty dataset
f = open(f"./{dataset_name}/{dataset_name}_reviews.json", 'w')
for l in parse(f"./{dataset_name}/reviews_{dataset_name}_5.json.gz"):
  f.write(l + '\n')

In [14]:
# print the number of lines in the file and the first line
data = open(f"./{dataset_name}/{dataset_name}_reviews.json", 'r')
print("Number of lines:", sum(1 for _ in data))
data.seek(0)  # Reset file pointer to the beginning
print("First line:", data.readline().strip())
data.close()

Number of lines: 198315
First line: {"reviewerID": "A1YJEY40YUW4SE", "asin": "7806397051", "reviewerName": "Andrea", "helpful": [3, 4], "reviewText": "Very oily and creamy. Not at all what I expected... ordered this to try to highlight and contour and it just looked awful!!! Plus, took FOREVER to arrive.", "overall": 1.0, "summary": "Don't waste your money", "unixReviewTime": 1391040000, "reviewTime": "01 30, 2014"}


In [16]:
import numpy as np
import pandas as pd

# Initialize mapping dictionaries
userID_mapping = {}
itemID_mapping = {}

# Open the JSON file for reading
data = open(f"./{dataset_name}/{dataset_name}_reviews.json", 'r')

# Initialize lists to store userID, itemID, and timestamp
userIDs = []
itemIDs = []
timestamps = []

# Process each line in the JSON file
for line in data:
    review = json.loads(line.strip())
    userID = review['reviewerID']
    itemID = review['asin']
    timestamp = review['unixReviewTime']
    
    # Map userID to an integer starting from 1
    if userID not in userID_mapping:
        userID_mapping[userID] = len(userID_mapping) + 1
    
    # Map itemID to an integer starting from 1
    if itemID not in itemID_mapping:
        itemID_mapping[itemID] = len(itemID_mapping) + 1
    
    # Append mapped values and timestamp to lists
    userIDs.append(userID_mapping[userID])
    itemIDs.append(itemID_mapping[itemID])
    timestamps.append(timestamp)

# Save mapping dictionaries as .npy files
np.save(f'./{dataset_name}/user_mapping.npy', userID_mapping)
print("user_num:", len(userID_mapping))
print("the first five userID mapping:", list(userID_mapping.items())[:5])
np.save(f'./{dataset_name}/item_mapping.npy', itemID_mapping)
print("item_num:", len(itemID_mapping))
print("the first five itemID mapping:", list(itemID_mapping.items())[:5])

# Group itemIDs by userID and sort by timestamp
user_item_mapping = {}
for userID, itemID, timestamp in zip(userIDs, itemIDs, timestamps):
    if userID not in user_item_mapping:
        user_item_mapping[userID] = []
    user_item_mapping[userID].append((itemID, timestamp))

# Sort itemIDs for each user by timestamp
for userID in user_item_mapping:
    user_item_mapping[userID].sort(key=lambda x: x[1])
    user_item_mapping[userID] = [item[0] for item in user_item_mapping[userID]]

user_item_mapping = {u: seq for u, seq in user_item_mapping.items() if len(seq) >= 3}

# Print a sample of the results
print("user-item mapping:", list(user_item_mapping.items())[:5])

# Split data into training, validation, and testing sets using leave-one-out strategy
train_data = {}
val_data = {}
test_data = {}

for userID, item_sequence in user_item_mapping.items():
    # Assign the last item for testing, the second-to-last for validation, and the rest for training
    train_data[userID] = item_sequence[:-2]
    val_data[userID] = item_sequence[:-1]
    test_data[userID] = item_sequence

# Print a sample of the split data
print("training data:", list(train_data.items())[:5])
print("validation data:", list(val_data.items())[:5])
print("testing data:", list(test_data.items())[:5])

# Prepare data for train, validation, and test sets
def prepare_data(data_dict):
    rows = []
    for userID, item_sequence in data_dict.items():
        history = item_sequence[:-1]
        target = item_sequence[-1]
        rows.append({'user': userID, 'history': history, 'target': target})
    return pd.DataFrame(rows)

# Create dataframes for train, validation, and test sets
train_df = prepare_data(train_data)
print("\nTraining data shape:", train_df.shape)
print("the first 3 rows of training data:\n", train_df.head(3))
val_df = prepare_data(val_data)
print("\nValidation data shape:", val_df.shape)
print("the first 3 rows of validation data:\n", val_df.head(3))
test_df = prepare_data(test_data)
print("\nTesting data shape:", test_df.shape)
print("the first 3 rows of testing data:\n", test_df.head(3))

# Save dataframes to parquet files
train_df.to_parquet(f'./{dataset_name}/train.parquet', index=False)
val_df.to_parquet(f'./{dataset_name}/valid.parquet', index=False)
test_df.to_parquet(f'./{dataset_name}/test.parquet', index=False)

print("Data saved to parquet files.")

data.close()


user_num: 22363
the first five userID mapping: [('A1YJEY40YUW4SE', 1), ('A60XNB876KYML', 2), ('A3G6XNM240RMWA', 3), ('A1PQFP6SAJ6D80', 4), ('A38FVHZTNQ271F', 5)]
item_num: 12083
the first five itemID mapping: [('7806397051', 1), ('9759091062', 2), ('9788072216', 3), ('9790790961', 4), ('9790794231', 5)]
user-item mapping: [(1, [6846, 7873, 4585, 1, 5406]), (2, [816, 10406, 11194, 11651, 9716, 1, 233]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204, 5863, 6609]), (4, [5522, 439, 5161, 11140, 1, 7849]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444, 11390])]
training data: [(1, [6846, 7873, 4585]), (2, [816, 10406, 11194, 11651, 9716]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204]), (4, [5522, 439, 5161, 11140]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500])]
validation data: [(1, [6846, 7873, 4585, 1]), (2, [816, 10406, 11194, 11651, 9716, 1]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204, 5863]), (4, [5522, 439, 5161, 11140, 1]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444]

# Generate Item Semantic Embeddings

In [17]:
f = open(f"./{dataset_name}/{dataset_name}_metadata.json", 'w')
for l in parse(f"./{dataset_name}/meta_{dataset_name}.json.gz"):
  f.write(l + '\n')

In [18]:
with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:

    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}

    item_info = {}
    
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')
        if asin in reverse_itemID_mapping.values():
            itemID = itemID_mapping[asin]
            item_info[itemID] = {
                'title': metadata.get('title') if metadata.get('title') else None,
                'description': metadata.get('description') if metadata.get('description') else None,
            }

for itemID, info in list(item_info.items())[:5]:
    print(f"ItemID: {itemID}, Info: {info}")

ItemID: 1, Info: {'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'description': 'An extensive range of 15 multiple vibrant long wear concealer colour with different skin tones to create more than 10,000 amazing looks. Using the most commonly applied shades, ensures the best skin colour match and guarantees a traceless and natural finish. Enabling layering and mixing, provides total camouflage for almost any skin problem including blemishes, scars, birthmarks and black circles. It is also suitable to use as bronzer. The light colour is suitable for redness, acne and so on. The medium colour is perfect for dark shadows in the under-eye area. The dark colour provides exceptional camouflage and adheres well to the skin. Silky glossy colour and high quality ingredients together to care skin around and can last for all day long. It is perfect for Professional Salon, Wedding, Party and Home use. Size: 15.4 x 10.2 x 1.3cm. Each Diameter: 2.6c

In [19]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer('sentence-transformers/sentence-t5-base')

# item_embeddings = []

# for i, (itemID, info) in enumerate(item_info.items()):
#     if i % 1000 == 0:
#         print(f"Processed {i}/{total_items}")
#     semantics = f"'title':{info.get('title', '')}\n 'description':{info.get('description', '')}"
#     embedding = model.encode(semantics)
#     item_embeddings.append({'ItemID': itemID, 'embedding': embedding.tolist()})

# item_emb_df = pd.DataFrame(item_embeddings)

# print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
# print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))

# output_path = f'./{dataset_name}/item_emb_td.parquet'
# item_emb_df.to_parquet(output_path, index=False)

# print("Item embeddings saved to item_emb.parquet.")

In [ ]:
from sentence_transformers import SentenceTransformer

# set local model path
local_model_path = ' '

print(f"Loading model from local path: {local_model_path} ...")

model = SentenceTransformer(local_model_path) # sentence-t5-base

item_embeddings = []

total_items = len(item_info)
print(f"Processing {total_items} items...")

for i, (itemID, info) in enumerate(item_info.items()):
    if i % 1000 == 0:
        print(f"Processed {i}/{total_items}")
    semantics = f"'title':{info.get('title', '')}\n 'description':{info.get('description', '')}"
    embedding = model.encode(semantics)
    item_embeddings.append({'ItemID': itemID, 'embedding': embedding.tolist()})

item_emb_df = pd.DataFrame(item_embeddings)

print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))

output_path = f'./{dataset_name}/item_emb_td.parquet'
item_emb_df.to_parquet(output_path, index=False)

print(f"Item embeddings saved to {output_path}.")

In [21]:
# Make sure the sequence is correct
parquet_path = f'./{dataset_name}/item_emb_td.parquet'

print("Target parquet:", parquet_path)

df = pd.read_parquet(parquet_path)
if "ItemID" not in df.columns:
    raise ValueError(f"'ItemID' column not found. columns={df.columns.tolist()}")

print("Loaded rows:", len(df), "cols:", df.columns.tolist())

df_sorted = df.sort_values("ItemID").reset_index(drop=True)
df_sorted.to_parquet(parquet_path, index=False)
print("Overwritten parquet with ItemID-sorted version.")

df_final = pd.read_parquet(parquet_path, columns=["ItemID"])
ids = df_final["ItemID"].to_numpy(dtype=np.int64)

non_decreasing = bool(np.all(ids[:-1] <= ids[1:])) if ids.size > 1 else True
strict_increasing = bool(np.all(ids[:-1] < ids[1:])) if ids.size > 1 else True
dense_1_to_n = bool(
    ids.size > 0 and
    ids.min() == 1 and
    ids.max() == ids.size and
    len(np.unique(ids)) == ids.size
)

print("=== CHECK ===")
print("rows:", ids.size)
print("first20:", ids[:20].tolist())
print("non_decreasing:", non_decreasing)
print("strict_increasing:", strict_increasing)
print("dense_1_to_n:", dense_1_to_n)

assert strict_increasing, "ItemID is not strictly increasing after overwrite!"
assert dense_1_to_n, "ItemID is not dense 1..N after overwrite!"
print("Verification passed.")

Target parquet: ./Beauty/item_emb_td.parquet
Loaded rows: 12083 cols: ['ItemID', 'embedding']
Overwritten parquet with ItemID-sorted version.
=== CHECK ===
rows: 12083
first20: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
non_decreasing: True
strict_increasing: True
dense_1_to_n: True
Verification passed.
